# Frozen YOLO-World reference on iWildCam-WILDS

This notebook will evaluate a pretrained, prompted YOLO-World model without updating its weights on WILDS. We will preserve the official camera-location splits, use training/validation data for setup and decisions, and leave both test splits untouched until final evaluation. The final task is one image-level WILDS class per photo, including empty images.

Tasks 1–3 cover Molab setup, inspection of the official iWildCam-WILDS data, and a frozen YOLO-World smoke test. No model training occurs in this notebook.

## Runtime and package inventory

Run the next cells in Molab and share their outputs before we continue to dataset loading. Missing packages are recorded here; we will install only those needed for the selected WILDS and YOLO-World workflow.

In [ ]:
# CELL 1
import platform
import sys
from importlib import metadata
from pathlib import Path

print('Python:', sys.version.splitlines()[0])
print('Platform:', platform.platform())
print('Working directory:', Path.cwd())

packages = ('torch', 'torchvision', 'wilds', 'mmyolo', 'mmdet', 'mmcv', 'numpy', 'Pillow')
for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = 'not installed'
    print(f'{package}: {version}')

In [ ]:
# CELL 2
import shutil

gib = 1024 ** 3
disk = shutil.disk_usage(Path.cwd())
print(f'Working-filesystem space: {disk.free / gib:.1f} GiB free / {disk.total / gib:.1f} GiB total')
print('This is a runtime snapshot, not a guarantee of persistent storage.')

try:
    import torch
except (ImportError, OSError) as exc:
    print('PyTorch could not be imported:', repr(exc))
else:
    print('PyTorch CUDA build:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        for device_index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(device_index)
            print(f'GPU {device_index}: {properties.name}; VRAM: {properties.total_memory / gib:.1f} GiB')

## Dataset location

Set DATA_ROOT to the parent folder of iwildcam_v2.0 if the dataset is already available elsewhere in Molab. This cell only checks the path; CELL 5 handles the official download.

In [ ]:
# CELL 3
DATA_ROOT = (Path.cwd() / 'data').resolve()  # Change if iWildCam is stored elsewhere.
DATASET_DIR = DATA_ROOT / 'iwildcam_v2.0'

print('WILDS root directory:', DATA_ROOT)
print('Expected iWildCam directory:', DATASET_DIR)
print('Dataset directory exists:', DATASET_DIR.is_dir())
print('metadata.csv exists:', (DATASET_DIR / 'metadata.csv').is_file())
print('categories.csv exists:', (DATASET_DIR / 'categories.csv').is_file())
if not (DATASET_DIR / 'metadata.csv').is_file():
    print('Dataset is not ready at this path; no download was attempted.')

## Task 2: Load and inspect iWildCam-WILDS

Run CELL 4 first, then CELL 5. The official CodaLab download currently fails TLS verification, so CELL 5 uses our teammate's [Hugging Face mirror](https://huggingface.co/datasets/emmilly/iwildcam-wilds-v2) of the **labeled** iWildCam v2.0 archive. It pins a repository revision and verifies the archive hash before extraction (about 12 GB download plus extracted data). The mirror's original CodaLab transfer did not verify TLS; these checks establish consistency with the mirror, not independent authenticity of the original source. The data may not persist across Molab sessions. If installation or download fails, stop and share that cell's output before continuing.

In [ ]:
# CELL 4
import subprocess

try:
    installed_wilds = metadata.version('wilds')
except metadata.PackageNotFoundError:
    installed_wilds = None
if installed_wilds != '2.0.0':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'wilds==2.0.0'], check=True)
try:
    installed_hub = metadata.version('huggingface-hub')
except metadata.PackageNotFoundError:
    installed_hub = None
if installed_hub is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub'], check=True)
import wilds
print('WILDS:', wilds.__version__)
print('huggingface_hub:', metadata.version('huggingface-hub'))

In [ ]:
# CELL 5
import hashlib
import tarfile

import pandas as pd
from huggingface_hub import hf_hub_download
from wilds import get_dataset

MIRROR_REPO = 'emmilly/iwildcam-wilds-v2'
MIRROR_REVISION = '1c01b321d444aa8a15e0bc7a0204d8c6559c8c5b'
ARCHIVE_SHA256 = '05c872f46eee44315f79bc90f4098c69ae3799a2b95116db02292ea1b4e370d2'
ARCHIVE_BYTES = 11_957_415_473
RELEASE_MARKER = DATASET_DIR / 'RELEASE_v2.0.txt'

DATA_ROOT.mkdir(parents=True, exist_ok=True)
required_files = (DATASET_DIR / 'metadata.csv', DATASET_DIR / 'categories.csv', RELEASE_MARKER)
if not (all(path.is_file() for path in required_files) and (DATASET_DIR / 'train').is_dir()):
    print('Downloading the pinned iWildCam v2.0 archive from', MIRROR_REPO)
    archive = Path(hf_hub_download(
        repo_id=MIRROR_REPO, filename='iwildcam_v2.0.tar.gz', repo_type='dataset',
        revision=MIRROR_REVISION, local_dir=str(DATA_ROOT),
    ))
    assert archive.stat().st_size == ARCHIVE_BYTES, 'Archive size differs from mirror manifest'
    digest = hashlib.sha256()
    with archive.open('rb') as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == ARCHIVE_SHA256, 'Archive SHA-256 differs from mirror manifest'
    print('Archive size and SHA-256 verified. Extracting into', DATASET_DIR)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        tar.extractall(DATASET_DIR, filter='data')

metadata_file = DATASET_DIR / 'metadata.csv'
assert metadata_file.is_file() and (DATASET_DIR / 'categories.csv').is_file()
assert (DATASET_DIR / 'train').is_dir(), 'Image directory is missing'
metadata_md5 = hashlib.md5(metadata_file.read_bytes()).hexdigest()
assert metadata_md5 == 'ab4905ee40ac45ab66ddb176d473a4b4', 'Metadata differs from mirror manifest'
split_counts = pd.read_csv(metadata_file, usecols=['split'])['split'].value_counts().to_dict()
expected_counts = {'train': 129809, 'val': 14961, 'test': 42791, 'id_val': 7314, 'id_test': 8154}
assert split_counts == expected_counts, f'Unexpected official split sizes: {split_counts}'
if not RELEASE_MARKER.is_file():
    RELEASE_MARKER.touch()  # WILDS recognizes a verified manual extraction.
dataset = get_dataset(dataset='iwildcam', version='2.0', root_dir=str(DATA_ROOT), download=False)
print('Dataset ready:', DATASET_DIR, '| classes:', dataset.n_classes)

### Official splits and labels

We list the five splits but inspect **only train, id_val, and val** labels and images. Leave id_test and test untouched for final evaluation. The class names come from the dataset's own categories.csv, not generated text.

In [ ]:
# CELL 6
SPLITS = ('train', 'id_val', 'val', 'id_test', 'test')
print('Number of classes:', dataset.n_classes)
print('Metadata fields:', dataset.metadata_fields)
def show_official_splits(wilds_dataset, split_names):
    assert set(split_names) == set(wilds_dataset.split_dict), 'Unexpected official split names'
    for split_name in split_names:
        subset_size = len(wilds_dataset.get_subset(split_name))  # No image or label is read.
        print(f'{split_name:7s} | {wilds_dataset.split_names[split_name]:22s} | {subset_size:,} images')

show_official_splits(dataset, SPLITS)

In [ ]:
# CELL 7
categories = pd.read_csv(DATASET_DIR / 'categories.csv')
print('Category columns:', list(categories.columns))
print(categories.head(5).to_string(index=True))
assert {'y', 'name'} <= set(categories.columns), 'Inspect categories.csv columns'
active_categories = categories[categories['y'].between(0, dataset.n_classes - 1)].copy()
print('Catalog rows outside the WILDS class range:', len(categories) - len(active_categories))
assert active_categories['y'].is_unique, 'Duplicate active WILDS class IDs'
id_to_name = dict(zip(active_categories['y'].astype(int), active_categories['name'].astype(str)))
assert set(id_to_name) == set(range(dataset.n_classes)), 'Active class IDs do not cover WILDS labels'
empty_ids = [i for i, name in id_to_name.items() if name.strip().lower() == 'empty']
assert len(empty_ids) == 1, f'Expected one empty class, found {empty_ids}'
empty_id = empty_ids[0]
print('Mapped classes:', len(id_to_name), '| empty class ID:', empty_id)
print('First 12 classes:', [(i, id_to_name[i]) for i in range(min(12, dataset.n_classes))])

In [ ]:
# CELL 8
from collections import Counter

def summarize_train_and_validation(wilds_dataset, names, empty_label, class_names):
    location_column = wilds_dataset.metadata_fields.index('location')
    subsets = {name: wilds_dataset.get_subset(name) for name in names}
    locations_by_split = {}
    for split_name, subset in subsets.items():
        labels = subset.y_array.tolist()
        locations = subset.metadata_array[:, location_column].tolist()
        locations_by_split[split_name] = set(locations)
        counts = Counter(labels)
        print(
            f'\n{split_name}: {len(subset):,} images, {len(counts)} observed classes, '
            f'{len(locations_by_split[split_name])} locations, '
            f'{counts[empty_label] / len(subset):.1%} empty'
        )
        print('Top classes:', [(class_names[i], n) for i, n in counts.most_common(8)])
    overlap = locations_by_split['train'] & locations_by_split['val']
    print('Train/OOD-val location overlap:', len(overlap))
    assert not overlap, 'OOD validation shares training locations'
    return subsets, location_column

inspected_splits, location_col = summarize_train_and_validation(
    dataset, ('train', 'id_val', 'val'), empty_id, id_to_name
)

### A few non-test photos

View one empty and one non-empty example from each training/validation split. These are image-level labels; they do not imply a single animal or provide bounding-box annotations.

In [ ]:
# CELL 9
import matplotlib.pyplot as plt

def show_examples(subsets, empty_label, class_names, location_column):
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for col, (split_name, subset) in enumerate(subsets.items()):
        labels = subset.y_array.tolist()
        for row, want_empty in enumerate((True, False)):
            sample_index = next(
                (i for i, y in enumerate(labels) if (y == empty_label) == want_empty), None
            )
            ax = axes[row, col]
            ax.axis('off')
            if sample_index is None:
                missing_kind = 'empty' if want_empty else 'animal'
                ax.set_title(f'{split_name}: no {missing_kind} sample')
                continue
            image, label, sample_metadata = subset[sample_index]
            ax.imshow(image.convert('RGB'))
            label_id = int(label)
            location_id = int(sample_metadata[location_column])
            ax.set_title(
                f'{split_name}: {class_names[label_id]} (ID {label_id})\n'
                f'{image.width}x{image.height}, location {location_id}'
            )
            print(
                f'{split_name} example: label={label_id}, size={image.size}, '
                f'location={location_id}'
            )
    plt.tight_layout()
    plt.show()

show_examples(inspected_splits, empty_id, id_to_name, location_col)

## Task 3: Frozen YOLO-World smoke test

We use Ultralytics' [YOLOv8s-Worldv2](https://docs.ultralytics.com/models/yolo-world/) implementation and `yolov8s-worldv2.pt` weights, which Ultralytics says were migrated from the [YOLO-World authors' release](https://github.com/AILab-CVC/YOLO-World). This is an implementation choice, not a claim that our notebook runs the authors' original MMYOLO code. CELLS 10–13 install the pinned package, download and hash the checkpoint, set a **temporary four-animal smoke-test vocabulary**, and inspect box-level predictions on a few train/validation images. The full 181-animal WILDS prompt mapping comes in Task 4. No test split or WILDS weight update is involved.

The first `set_classes` call may fetch CLIP text-encoder dependencies or weights. Stop and share the cell output if this or checkpoint loading fails.

In [ ]:
# CELL 10
ULTRALYTICS_VERSION = '8.4.178'
try:
    installed_ultralytics = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    installed_ultralytics = None
if installed_ultralytics != ULTRALYTICS_VERSION:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', f'ultralytics=={ULTRALYTICS_VERSION}'],
        check=True,
    )
import ultralytics
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Task 3 expects the Molab GPU'

In [ ]:
# CELL 11
from ultralytics import YOLOWorld

WORLD_WEIGHTS = 'yolov8s-worldv2.pt'
WORLD_SOURCE = (
    'https://github.com/ultralytics/assets/releases/download/v8.4.0/'
    'yolov8s-worldv2.pt'
)
WORLD_IMAGE_SIZE = 640
world_model = YOLOWorld(WORLD_WEIGHTS)  # Downloads the checkpoint if absent.
world_model.model.eval()
world_model.model.requires_grad_(False)
world_checkpoint = Path(WORLD_WEIGHTS).resolve()
assert world_checkpoint.is_file(), f'Checkpoint not found at {world_checkpoint}'
world_digest = hashlib.sha256()
with world_checkpoint.open('rb') as world_file:
    for world_chunk in iter(lambda: world_file.read(8 * 1024 * 1024), b''):
        world_digest.update(world_chunk)
WORLD_SHA256 = world_digest.hexdigest()
print('Checkpoint:', world_checkpoint)
print('Source:', WORLD_SOURCE)
print('Bytes:', world_checkpoint.stat().st_size, '| SHA-256:', WORLD_SHA256)
print('Input setting: letterbox to', WORLD_IMAGE_SIZE, 'pixels; Ultralytics handles scaling')
print('Trainable parameters:', sum(p.numel() for p in world_model.model.parameters() if p.requires_grad))

In [ ]:
# CELL 12
SMOKE_PROMPTS = ['cow', 'ocelot', 'fox', 'deer']
world_model.set_classes(SMOKE_PROMPTS)  # Encodes prompts; does not train on WILDS.
world_model.model.eval()
world_model.model.requires_grad_(False)
assert [world_model.names[i] for i in range(len(SMOKE_PROMPTS))] == SMOKE_PROMPTS
print('Temporary smoke-test prompts:', world_model.names)
print('Trainable parameters:', sum(p.numel() for p in world_model.model.parameters() if p.requires_grad))

In [ ]:
# CELL 13
import time

def smoke_test_world(model, subsets, class_names, location_column):
    cases = (('train', 24), ('id_val', 8), ('val', 145), ('val', 0))
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    torch.cuda.reset_peak_memory_stats()
    for ax, (split_name, target_label) in zip(axes.flat, cases):
        subset = subsets[split_name]
        sample_index = next(
            (i for i, y in enumerate(subset.y_array.tolist()) if y == target_label), None
        )
        assert sample_index is not None, f'No label {target_label} in {split_name}'
        image, label, sample_metadata = subset[sample_index]
        started = time.perf_counter()
        with torch.inference_mode():
            result = model.predict(
                source=image.convert('RGB'), imgsz=WORLD_IMAGE_SIZE, conf=0.05,
                iou=0.7, max_det=20, device=0, verbose=False, save=False,
            )[0]
        elapsed = time.perf_counter() - started
        assert tuple(result.orig_shape) == (image.height, image.width)
        box_rows = []
        for box in result.boxes:
            class_index = int(box.cls.item())
            score = float(box.conf.item())
            xyxy = [round(float(v), 1) for v in box.xyxy[0].tolist()]
            assert 0 <= class_index < len(SMOKE_PROMPTS)
            assert 0 <= xyxy[0] <= xyxy[2] <= image.width + 1
            assert 0 <= xyxy[1] <= xyxy[3] <= image.height + 1
            box_rows.append((SMOKE_PROMPTS[class_index], round(score, 3), xyxy))
        location = int(sample_metadata[location_column])
        print(
            f'{split_name}: true={class_names[int(label)]}, location={location}, '
            f'boxes={len(box_rows)}, seconds={elapsed:.2f}'
        )
        print('  top boxes:', sorted(box_rows, key=lambda item: -item[1])[:5])
        ax.imshow(result.plot()[..., ::-1])  # Ultralytics plot is BGR; matplotlib is RGB.
        ax.set_title(f'{split_name}: {class_names[int(label)]}')
        ax.axis('off')
    print('Peak allocated GPU memory:', round(torch.cuda.max_memory_allocated() / gib, 2), 'GiB')
    plt.tight_layout()
    plt.show()

smoke_test_world(world_model, inspected_splits, id_to_name, location_col)

## Task 4: WILDS animal prompts and label mapping
Use each active non-empty WILDS class name verbatim as the initial text prompt. These are dataset labels, not generated captions or verified English common names. One non-empty class is `motorcycle`, so the prompts are not exclusively animals. `empty` is excluded: it will be the image-level no-detection outcome in Task 5. Inspect Cell 14's complete mapping before running Cell 15. This prompt set is frozen for the initial validation reference; any alternative wording will be a separately recorded validation-only experiment.

In [ ]:
# CELL 14
PROMPT_TEMPLATE = '{label}'  # Exact active WILDS category name; no generated synonyms.
animal_class_ids = tuple(i for i in range(dataset.n_classes) if i != empty_id)
animal_prompts = tuple(
    PROMPT_TEMPLATE.format(label=id_to_name[y].strip()) for y in animal_class_ids
)
prompt_index_to_y = dict(enumerate(animal_class_ids))
y_to_prompt_index = {y: i for i, y in prompt_index_to_y.items()}
assert len(animal_prompts) == dataset.n_classes - 1
assert all(animal_prompts) and len(set(animal_prompts)) == len(animal_prompts), 'Blank or duplicate animal prompts'
assert empty_id not in y_to_prompt_index
assert all(
    y_to_prompt_index[prompt_index_to_y[i]] == i
    and animal_prompts[i] == id_to_name[prompt_index_to_y[i]].strip()
    for i in range(len(animal_prompts))
)
prompt_table = pd.DataFrame({
    'detector_index': range(len(animal_prompts)),
    'wilds_y': animal_class_ids,
    'dataset_name_and_prompt': animal_prompts,
})
print('Prompt template:', PROMPT_TEMPLATE, '| non-empty class prompts:', len(animal_prompts))
print('Image-level no-animal label:', empty_id, id_to_name[empty_id])
print(prompt_table.to_string(index=False))

Cell 15 replaces the four temporary smoke-test prompts in the frozen model with all non-empty WILDS class prompts. If you later rerun Cell 13, rerun Cell 12 first to restore its temporary prompts, then rerun Cell 15 before continuing. A missing box in this one-image check is not a failure of the index mapping.

In [ ]:
# CELL 15
world_model.set_classes(list(animal_prompts))
world_model.model.eval()
world_model.model.requires_grad_(False)
assert len(world_model.names) == len(animal_prompts)
assert all(world_model.names[i] == animal_prompts[i] for i in range(len(animal_prompts)))
assert sum(p.numel() for p in world_model.model.parameters() if p.requires_grad) == 0
print('Frozen model configured with', len(animal_prompts), 'non-empty WILDS class prompts')
print('Detector index 0 -> WILDS y', prompt_index_to_y[0], '->', world_model.names[0])
print('Detector index', len(animal_prompts) - 1, '-> WILDS y',
      prompt_index_to_y[len(animal_prompts) - 1], '->', world_model.names[len(animal_prompts) - 1])

# One non-test image checks actual returned class indices without estimating performance.
mapping_subset = inspected_splits['train']
mapping_index = next(i for i, y in enumerate(mapping_subset.y_array.tolist()) if int(y) == 24)
mapping_image, mapping_true_y, _ = mapping_subset[mapping_index]
with torch.inference_mode():
    mapping_result = world_model.predict(
        source=mapping_image.convert('RGB'), imgsz=WORLD_IMAGE_SIZE, conf=0.05,
        iou=0.7, max_det=20, device=0, verbose=False, save=False,
    )[0]
mapped_detections = []
for box in mapping_result.boxes:
    detector_index = int(box.cls.item())
    assert detector_index in prompt_index_to_y
    wilds_y = prompt_index_to_y[detector_index]
    mapped_detections.append((wilds_y, id_to_name[wilds_y], round(float(box.conf.item()), 3)))
print('Example true WILDS label:', int(mapping_true_y), id_to_name[int(mapping_true_y)])
print('Mapped detections (top five):', sorted(mapped_detections, key=lambda row: -row[2])[:5])
if not mapped_detections:
    print('No detections at conf=0.05; prompt-index mapping still passed the checks above.')

## Task 5: One image-level label from detections
The fixed rule is highest-scoring non-empty detection; if no detection exists or its score is below a threshold, predict WILDS `empty` (ID 0). Tied top scores go to the smaller WILDS ID, then the earlier box. The threshold is selected on held-out-camera validation, not on either test split. A low detector confidence floor preserves candidates for threshold selection; it is not the selected image-level threshold.

In [ ]:
# CELL 16
DETECTION_FLOOR = 0.001  # Detector filtering only; image-level threshold is selected later.
DETECTION_IOU = 0.7
DETECTION_MAX_BOXES = 20

def extract_candidate_boxes(result, detector_to_y):
    candidates = []
    for box_index, box in enumerate(result.boxes):
        detector_index = int(box.cls.item())
        if detector_index not in detector_to_y:
            raise ValueError(f'Unknown detector class index: {detector_index}')
        candidates.append({
            'box_index': box_index,
            'detector_index': detector_index,
            'wilds_y': detector_to_y[detector_index],
            'score': float(box.conf.item()),
            'xyxy': [float(v) for v in box.xyxy[0].tolist()],
        })
    return candidates

def choose_image_label(candidates, threshold, empty_label):
    if not 0.0 <= threshold <= 1.0:
        raise ValueError('Threshold must be between 0 and 1')
    if not candidates:
        return {'pred_y': empty_label, 'top': None, 'reason': 'no_boxes'}
    top = min(candidates, key=lambda row: (
        -row['score'], row['wilds_y'], row['box_index']
    ))
    if top['score'] < threshold:
        return {'pred_y': empty_label, 'top': top, 'reason': 'below_threshold'}
    return {'pred_y': top['wilds_y'], 'top': top, 'reason': 'accepted_detection'}

def check_image_label_edge_cases():
    empty = 0
    a = {'wilds_y': 24, 'score': 0.6, 'box_index': 1}
    b = {'wilds_y': 41, 'score': 0.8, 'box_index': 0}
    c = {'wilds_y': 8, 'score': 0.8, 'box_index': 2}
    assert choose_image_label([], 0.2, empty)['reason'] == 'no_boxes'
    assert choose_image_label([a], 0.7, empty)['pred_y'] == empty
    assert choose_image_label([a], 0.6, empty)['pred_y'] == 24
    assert choose_image_label([a, b], 0.2, empty)['pred_y'] == 41
    assert choose_image_label([a, b, c], 0.2, empty)['pred_y'] == 8
    earlier_a = {**a, 'box_index': 0}
    assert choose_image_label([a, earlier_a], 0.2, empty)['top']['box_index'] == 0
    print('Adapter edge checks passed: no boxes, cutoff, equality, multiple boxes, ties.')

check_image_label_edge_cases()

In [ ]:
# CELL 17
def check_adapter_on_examples(model, subsets, detector_to_y, names, empty_label):
    # Illustrative only: 0.10 is not the selected validation threshold.
    provisional_threshold = 0.10
    cases = (('train', 24), ('id_val', 8), ('val', 145), ('val', empty_label))
    for split_name, target_y in cases:
        subset = subsets[split_name]
        sample_index = next(
            i for i, y in enumerate(subset.y_array.tolist()) if int(y) == target_y
        )
        image, true_y, _ = subset[sample_index]
        with torch.inference_mode():
            result = model.predict(
                source=image.convert('RGB'), imgsz=WORLD_IMAGE_SIZE,
                conf=DETECTION_FLOOR, iou=DETECTION_IOU,
                max_det=DETECTION_MAX_BOXES, device=0, verbose=False, save=False,
            )[0]
        candidates = extract_candidate_boxes(result, detector_to_y)
        selected = choose_image_label(candidates, provisional_threshold, empty_label)
        top_score = None if selected['top'] is None else round(selected['top']['score'], 3)
        print(
            f'{split_name}: true={int(true_y)} {names[int(true_y)]}; '
            f'pred={selected["pred_y"]} {names[selected["pred_y"]]}; '
            f'boxes={len(candidates)}; top_score={top_score}; reason={selected["reason"]}'
        )

check_adapter_on_examples(
    world_model, inspected_splits, prompt_index_to_y, id_to_name, empty_id
)

### Cache held-out-camera validation detections
Cell 18 is the first full validation inference run and may take a while. After reviewing Cells 16–17, `RUN_FULL_VAL_CACHE=True` is now set. It processes only official `val` (14,961 images), writes complete chunks under `DATA_ROOT`, and resumes by skipping verified chunks. Each cached row retains the true image label, camera location, and up to 20 candidate boxes with scores. The cache key includes the checkpoint, prompts, and inference settings. Neither `id_test` nor `test` is read.

In [ ]:
# CELL 18
import json
from tqdm.auto import tqdm

RUN_FULL_VAL_CACHE = True  # Cells 16-17 reviewed; run resumable validation cache.
VAL_BATCH_SIZE = 8
VAL_CHUNK_SIZE = 128

def cache_held_out_validation(
    model, subset, detector_to_y, prompts, location_column, data_root
):
    config = {
        'split': 'val', 'subset_size': len(subset),
        'checkpoint_sha256': WORLD_SHA256,
        'ultralytics_version': ultralytics.__version__,
        'prompts': list(prompts),
        'image_size': WORLD_IMAGE_SIZE, 'confidence_floor': DETECTION_FLOOR,
        'iou': DETECTION_IOU, 'max_boxes': DETECTION_MAX_BOXES,
    }
    signature = hashlib.sha256(
        json.dumps(config, sort_keys=True).encode('utf-8')
    ).hexdigest()[:16]
    cache_dir = data_root / f'yolo_world_val_{signature}'
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = cache_dir / 'manifest.json'
    if manifest_path.is_file():
        assert json.loads(manifest_path.read_text(encoding='utf-8')) == config
    else:
        manifest_path.write_text(
            json.dumps(config, indent=2, sort_keys=True), encoding='utf-8'
        )
    assert all(model.names[i] == prompts[i] for i in range(len(prompts)))
    for start in tqdm(range(0, len(subset), VAL_CHUNK_SIZE), desc='val chunks'):
        stop = min(start + VAL_CHUNK_SIZE, len(subset))
        chunk_path = cache_dir / f'part_{start:05d}_{stop:05d}.jsonl'
        if chunk_path.is_file():
            saved = [json.loads(line) for line in chunk_path.read_text(
                encoding='utf-8'
            ).splitlines()]
            assert [row['index'] for row in saved] == list(range(start, stop)), (
                f'Incomplete cache chunk: {chunk_path}'
            )
            continue
        rows = []
        for batch_start in range(start, stop, VAL_BATCH_SIZE):
            batch_stop = min(batch_start + VAL_BATCH_SIZE, stop)
            examples = [subset[i] for i in range(batch_start, batch_stop)]
            images = [image.convert('RGB') for image, _, _ in examples]
            with torch.inference_mode():
                results = model.predict(
                    source=images, imgsz=WORLD_IMAGE_SIZE,
                    conf=DETECTION_FLOOR, iou=DETECTION_IOU,
                    max_det=DETECTION_MAX_BOXES, device=0,
                    verbose=False, save=False,
                )
            assert len(results) == len(examples)
            for offset, ((image, true_y, sample_metadata), result) in enumerate(
                zip(examples, results)
            ):
                assert tuple(result.orig_shape) == (image.height, image.width)
                rows.append({
                    'index': batch_start + offset,
                    'true_y': int(true_y),
                    'location': int(sample_metadata[location_column]),
                    'boxes': extract_candidate_boxes(result, detector_to_y),
                })
        assert [row['index'] for row in rows] == list(range(start, stop))
        temporary_path = chunk_path.with_suffix('.jsonl.tmp')
        temporary_path.write_text(
            ''.join(json.dumps(row, separators=(',', ':')) + '\n' for row in rows),
            encoding='utf-8',
        )
        temporary_path.replace(chunk_path)
    print('Cached', len(subset), 'held-out-camera validation images at', cache_dir)
    return cache_dir

if RUN_FULL_VAL_CACHE:
    val_cache_dir = cache_held_out_validation(
        world_model, inspected_splits['val'], prompt_index_to_y, animal_prompts,
        location_col, DATA_ROOT,
    )
else:
    val_cache_dir = None
    print('Full validation inference is off. Set RUN_FULL_VAL_CACHE=True when ready.')

### Select the image-level `empty` threshold
Cell 19 sweeps fixed score values and score quantiles, maximizing macro-F1 on official `val` using only classes present in its ground-truth labels, matching the WILDS evaluator. Ties select the lower threshold. This is **threshold tuning on validation**, so its best validation score is not an independent performance estimate. Task 6 will report diagnostics and apply the frozen threshold to `id_val`; final test remains untouched.

In [ ]:
# CELL 19
import numpy as np
from sklearn.metrics import f1_score

def select_empty_threshold(cache_dir, subset, empty_label):
    records = []
    for start in range(0, len(subset), VAL_CHUNK_SIZE):
        stop = min(start + VAL_CHUNK_SIZE, len(subset))
        chunk_path = cache_dir / f'part_{start:05d}_{stop:05d}.jsonl'
        assert chunk_path.is_file(), f'Missing validation chunk: {chunk_path}'
        records.extend(json.loads(line) for line in chunk_path.read_text(
            encoding='utf-8'
        ).splitlines())
    assert len(records) == len(subset)
    assert [row['index'] for row in records] == list(range(len(subset)))
    true_y = np.array([row['true_y'] for row in records], dtype=np.int64)
    assert np.array_equal(true_y, subset.y_array.numpy()), 'Cache labels mismatch'
    tops = [choose_image_label(row['boxes'], 0.0, empty_label)['top']
            for row in records]
    top_y = np.array([empty_label if top is None else top['wilds_y']
                      for top in tops], dtype=np.int64)
    top_score = np.array([-1.0 if top is None else top['score']
                          for top in tops], dtype=np.float64)
    positive_scores = top_score[top_score >= 0]
    candidates = {0.0, 1.0}
    candidates.update(float(x) for x in np.linspace(0.01, 0.99, 99))
    if len(positive_scores):
        candidates.update(float(x) for x in np.quantile(
            positive_scores, np.linspace(0, 1, 101)
        ))
    true_labels = sorted(set(true_y.tolist()))
    sweep = []
    for threshold in sorted(candidates):
        predicted = np.where(top_score >= threshold, top_y, empty_label)
        sweep.append({
            'threshold': threshold,
            'val_macro_f1': f1_score(
                true_y, predicted, average='macro', labels=true_labels,
                zero_division=0,
            ),
            'val_accuracy': float(np.mean(predicted == true_y)),
            'predicted_empty_fraction': float(np.mean(predicted == empty_label)),
        })
    sweep_table = pd.DataFrame(sweep).sort_values(
        ['val_macro_f1', 'threshold'], ascending=[False, True]
    ).reset_index(drop=True)
    selected_threshold = float(sweep_table.iloc[0]['threshold'])
    selected_y = np.where(top_score >= selected_threshold, top_y, empty_label)
    assert all(
        choose_image_label(row['boxes'], selected_threshold, empty_label)['pred_y']
        == int(selected_y[i]) for i, row in enumerate(records)
    )
    print('Validation images:', len(records), '| no boxes:', int(np.sum(top_score < 0)))
    print('Threshold candidates:', len(sweep_table), '| tuning objective: WILDS-style macro-F1')
    print(sweep_table.head(10).to_string(index=False))
    print('Selected threshold:', selected_threshold)
    print('The selected validation F1 is a tuning score, not independent evaluation.')
    return selected_threshold, sweep_table, records

if val_cache_dir is None:
    CALIBRATED_EMPTY_THRESHOLD = None
    threshold_sweep = None
    val_detection_records = None
    print('Threshold selection waits for Cell 18 full-validation inference.')
else:
    CALIBRATED_EMPTY_THRESHOLD, threshold_sweep, val_detection_records = (
        select_empty_threshold(val_cache_dir, inspected_splits['val'], empty_id)
    )

## Task 6: Evaluate and diagnose the frozen reference
Cell 20 caches official `id_val` (familiar-camera validation) using the same frozen checkpoint, 181 non-empty prompts, image size, detector floor, IoU, and box limit as `val`. It resumes complete chunks if interrupted. The image-level threshold remains the value selected on `val` in Cell 19. No test image or label is read.

In [ ]:
# CELL 20
def cache_familiar_validation(model, subset, detector_to_y, prompts, location_column):
    assert CALIBRATED_EMPTY_THRESHOLD is not None, 'Run Cell 19 first'
    val_config = json.loads((val_cache_dir / 'manifest.json').read_text(encoding='utf-8'))
    assert val_config['checkpoint_sha256'] == WORLD_SHA256
    assert val_config['prompts'] == list(prompts)
    assert val_config['image_size'] == WORLD_IMAGE_SIZE
    assert val_config['confidence_floor'] == DETECTION_FLOOR
    assert val_config['iou'] == DETECTION_IOU
    assert val_config['max_boxes'] == DETECTION_MAX_BOXES
    assert all(model.names[i] == prompts[i] for i in range(len(prompts)))
    config = {**val_config, 'split': 'id_val', 'subset_size': len(subset)}
    signature = hashlib.sha256(
        json.dumps(config, sort_keys=True).encode('utf-8')
    ).hexdigest()[:16]
    cache_dir = DATA_ROOT / f'yolo_world_id_val_{signature}'
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = cache_dir / 'manifest.json'
    if manifest_path.is_file():
        assert json.loads(manifest_path.read_text(encoding='utf-8')) == config
    else:
        manifest_path.write_text(json.dumps(config, indent=2, sort_keys=True), encoding='utf-8')
    rows_all = []
    newly_inferred = 0
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for start in tqdm(range(0, len(subset), VAL_CHUNK_SIZE), desc='id_val chunks'):
        stop = min(start + VAL_CHUNK_SIZE, len(subset))
        chunk_path = cache_dir / f'part_{start:05d}_{stop:05d}.jsonl'
        if chunk_path.is_file():
            rows = [json.loads(line) for line in chunk_path.read_text(
                encoding='utf-8'
            ).splitlines()]
            assert [row['index'] for row in rows] == list(range(start, stop))
            rows_all.extend(rows)
            continue
        rows = []
        for batch_start in range(start, stop, VAL_BATCH_SIZE):
            batch_stop = min(batch_start + VAL_BATCH_SIZE, stop)
            examples = [subset[i] for i in range(batch_start, batch_stop)]
            images = [image.convert('RGB') for image, _, _ in examples]
            with torch.inference_mode():
                results = model.predict(
                    source=images, imgsz=WORLD_IMAGE_SIZE,
                    conf=DETECTION_FLOOR, iou=DETECTION_IOU,
                    max_det=DETECTION_MAX_BOXES, device=0,
                    verbose=False, save=False,
                )
            assert len(results) == len(examples)
            newly_inferred += len(examples)
            for offset, ((image, true_y, sample_metadata), result) in enumerate(
                zip(examples, results)
            ):
                assert tuple(result.orig_shape) == (image.height, image.width)
                rows.append({
                    'index': batch_start + offset,
                    'true_y': int(true_y),
                    'location': int(sample_metadata[location_column]),
                    'boxes': extract_candidate_boxes(result, detector_to_y),
                })
        assert [row['index'] for row in rows] == list(range(start, stop))
        temporary_path = chunk_path.with_suffix('.jsonl.tmp')
        temporary_path.write_text(
            ''.join(json.dumps(row, separators=(',', ':')) + '\n' for row in rows),
            encoding='utf-8',
        )
        temporary_path.replace(chunk_path)
        rows_all.extend(rows)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    assert [row['index'] for row in rows_all] == list(range(len(subset)))
    assert [row['true_y'] for row in rows_all] == subset.y_array.tolist()
    peak_gib = torch.cuda.max_memory_allocated() / gib
    print('Cached', len(rows_all), 'familiar-camera validation images at', cache_dir)
    print(f'This invocation: {newly_inferred} new images, {elapsed:.1f} s wall time, '
          f'{peak_gib:.2f} GiB peak PyTorch GPU allocation')
    return cache_dir, rows_all

id_val_cache_dir, id_val_detection_records = cache_familiar_validation(
    world_model, inspected_splits['id_val'], prompt_index_to_y,
    animal_prompts, location_col,
)

### Official image-level metrics and error breakdowns
Cell 21 applies the **same** `val`-selected threshold to both validation splits. It calls the iWildCam-WILDS evaluator for official image-level macro-F1 and accuracy, then prints empty/non-empty errors, frequent-class F1, common wrong-species pairs, and camera-location summaries. The `val` score is selection-tuned; `id_val` is a different, familiar-camera diagnostic, not a substitute for held-out-camera test performance.

In [ ]:
# CELL 21
from sklearn.metrics import precision_recall_fscore_support

def describe_validation_split(split_name, subset, records, threshold):
    assert len(records) == len(subset)
    assert [row['index'] for row in records] == list(range(len(subset)))
    true_y = np.array([row['true_y'] for row in records], dtype=np.int64)
    locations = np.array([row['location'] for row in records], dtype=np.int64)
    assert np.array_equal(true_y, subset.y_array.numpy())
    assert np.array_equal(
        locations, subset.metadata_array[:, location_col].numpy()
    )
    decisions = [choose_image_label(row['boxes'], threshold, empty_id)
                 for row in records]
    predicted_y = np.array([decision['pred_y'] for decision in decisions],
                           dtype=np.int64)
    tops = [decision['top'] for decision in decisions]
    diagnostic = pd.DataFrame({
        'index': range(len(records)), 'location': locations,
        'true_y': true_y, 'pred_y': predicted_y,
        'top_score': [-1.0 if top is None else top['score'] for top in tops],
        'box_count': [len(row['boxes']) for row in records],
        'reason': [decision['reason'] for decision in decisions],
    })
    diagnostic['correct'] = diagnostic['true_y'] == diagnostic['pred_y']
    diagnostic['predicted_empty'] = diagnostic['pred_y'] == empty_id
    _, official_text = dataset.eval(
        torch.as_tensor(predicted_y, dtype=torch.long),
        subset.y_array, subset.metadata_array,
    )
    true_empty = true_y == empty_id
    predicted_empty = predicted_y == empty_id
    print(f'\n{split_name}: {len(records):,} images; threshold={threshold:.6f}')
    print(official_text.strip())
    print(f'True empty: {np.mean(true_empty):.1%}; predicted empty: '
          f'{np.mean(predicted_empty):.1%}; no boxes: '
          f'{np.mean(diagnostic["box_count"] == 0):.1%}; '
          f'at {DETECTION_MAX_BOXES}-box cap: '
          f'{np.mean(diagnostic["box_count"] == DETECTION_MAX_BOXES):.1%}')
    print('Mean boxes/image:', round(float(diagnostic['box_count'].mean()), 2))
    print('True non-empty -> empty:', int(np.sum(~true_empty & predicted_empty)),
          'of', int(np.sum(~true_empty)))
    print('True empty -> non-empty:', int(np.sum(true_empty & ~predicted_empty)),
          'of', int(np.sum(true_empty)))
    print('Non-empty -> wrong non-empty class:', int(np.sum(
        ~true_empty & ~predicted_empty & (true_y != predicted_y)
    )))
    print('Correct non-empty:', int(np.sum(
        ~true_empty & (true_y == predicted_y)
    )))
    present_labels = sorted(set(true_y.tolist()))
    _, _, class_f1, support = precision_recall_fscore_support(
        true_y, predicted_y, labels=present_labels, zero_division=0
    )
    by_class = pd.DataFrame({
        'y': present_labels, 'name': [id_to_name[y] for y in present_labels],
        'support': support, 'f1': class_f1,
    }).sort_values(['support', 'y'], ascending=[False, True])
    print('Frequent classes (support and F1):')
    print(by_class.head(12).to_string(index=False))
    wrong_species = diagnostic.loc[
        (diagnostic['true_y'] != empty_id)
        & (diagnostic['pred_y'] != empty_id)
        & (diagnostic['true_y'] != diagnostic['pred_y'])
    ]
    pairs = wrong_species.groupby(['true_y', 'pred_y']).size().sort_values(
        ascending=False
    ).head(10)
    print('Most frequent wrong non-empty class pairs:')
    for (actual, guessed), count in pairs.items():
        print(f'  {id_to_name[actual]} ({actual}) -> '
              f'{id_to_name[guessed]} ({guessed}): {count}')
    by_location = diagnostic.groupby('location').agg(
        images=('index', 'size'), accuracy=('correct', 'mean'),
        predicted_empty=('predicted_empty', 'mean'),
    )
    eligible = by_location[by_location['images'] >= 30].sort_values('accuracy')
    print('Locations with >=30 images:', len(eligible))
    print('Lowest-accuracy locations:')
    print(eligible.head(8).to_string())
    print('Highest-accuracy locations:')
    print(eligible.tail(8).to_string())
    return diagnostic

id_val_diagnostics = describe_validation_split(
    'id_val (familiar cameras)', inspected_splits['id_val'],
    id_val_detection_records, CALIBRATED_EMPTY_THRESHOLD,
)
val_diagnostics = describe_validation_split(
    'val (held-out cameras; threshold-tuned)', inspected_splits['val'],
    val_detection_records, CALIBRATED_EMPTY_THRESHOLD,
)

### Representative held-out-camera examples
Cell 22 selects deterministic examples from the cached `val` diagnostics: wrong non-empty class, animal classified as `empty`, false non-empty detection on `empty`, and correct cases when available. It draws only the selected top box; the cache retains the other candidate boxes. Example selection is for understanding failure modes, not another prompt- or threshold-tuning pass.

In [ ]:
# CELL 22
from matplotlib.patches import Rectangle

def show_held_out_examples(subset, records, diagnostic, threshold):
    assert len(records) == len(diagnostic) == len(subset)
    actual = diagnostic['true_y']
    guessed = diagnostic['pred_y']
    cases = (
        ('wrong class', (actual != empty_id) & (guessed != empty_id) & (actual != guessed)),
        ('animal -> empty', (actual != empty_id) & (guessed == empty_id)),
        ('empty -> class', (actual == empty_id) & (guessed != empty_id)),
        ('correct class', (actual != empty_id) & (actual == guessed)),
        ('correct empty', (actual == empty_id) & (actual == guessed)),
    )
    selected = []
    for title, mask in cases:
        options = diagnostic.loc[mask].sort_values(
            ['top_score', 'index'], ascending=[False, True]
        )
        if not options.empty:
            selected.append((title, int(options.iloc[0]['index'])))
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for ax in axes.flat:
        ax.axis('off')
    for ax, (case_title, index) in zip(axes.flat, selected):
        record = records[index]
        image, true_y, _ = subset[index]
        decision = choose_image_label(record['boxes'], threshold, empty_id)
        predicted_y = decision['pred_y']
        assert int(true_y) == record['true_y']
        ax.imshow(image.convert('RGB'))
        top = decision['top']
        if top is not None:
            x1, y1, x2, y2 = top['xyxy']
            color = 'lime' if predicted_y == int(true_y) else (
                'orange' if predicted_y == empty_id else 'red'
            )
            ax.add_patch(Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor=color, linewidth=2,
            ))
        score_text = 'none' if top is None else f'{top["score"]:.3f}'
        ax.set_title(
            f'{case_title} | val index {index} | top score {score_text}\n'
            f'true: {id_to_name[int(true_y)]} ({int(true_y)}) | '
            f'pred: {id_to_name[predicted_y]} ({predicted_y})',
            fontsize=9,
        )
        print(f'{case_title}: val index={index}, true={int(true_y)}, '
              f'pred={predicted_y}, top_score={score_text}, '
              f'boxes={len(record["boxes"])}')
    fig.suptitle('Held-out-camera validation examples (top cached box only)')
    plt.tight_layout()
    plt.show()

show_held_out_examples(
    inspected_splits['val'], val_detection_records,
    val_diagnostics, CALIBRATED_EMPTY_THRESHOLD,
)

## Task 7: Freeze and hand off the reference
Cell 23 writes a compact, content-addressed JSON manifest under `DATA_ROOT`. It records the checkpoint and dataset provenance, exact ordered prompts and WILDS IDs, inference and image-level decision rules, selected threshold, official validation metrics, runtime environment, and known limitations. The JSON is the handoff contract for the later trained notebook. Download or copy this small file when moving to another Molab session; the large image data and box caches stay outside Git.

In [ ]:
# CELL 23
def build_frozen_reference_manifest():
    assert CALIBRATED_EMPTY_THRESHOLD is not None
    assert sum(p.numel() for p in world_model.model.parameters() if p.requires_grad) == 0
    selected_row = threshold_sweep.iloc[0]
    assert float(selected_row['threshold']) == CALIBRATED_EMPTY_THRESHOLD

    def package_version(distribution_name):
        try:
            return metadata.version(distribution_name)
        except metadata.PackageNotFoundError:
            return None

    try:
        clip_source_text = metadata.distribution('clip').read_text('direct_url.json')
    except metadata.PackageNotFoundError:
        clip_source_text = None
    clip_source = json.loads(clip_source_text) if clip_source_text else None
    versions = {
        'python': sys.version.split()[0],
        'torch_loaded': torch.__version__,
        'torch_installed': package_version('torch'),
        'torch_cuda_build': torch.version.cuda,
        'torchvision_installed': package_version('torchvision'),
        'ultralytics': ultralytics.__version__,
        'wilds': package_version('wilds'),
        'clip': package_version('clip'),
        'clip_direct_url': clip_source,
        'numpy': np.__version__,
        'scikit_learn': package_version('scikit-learn'),
        'pillow': package_version('Pillow'),
        'gpu_name': torch.cuda.get_device_properties(0).name,
        'gpu_total_gib': round(torch.cuda.get_device_properties(0).total_memory / gib, 2),
    }
    validation = {}
    for split_name, diagnostics in (
        ('id_val', id_val_diagnostics), ('val', val_diagnostics)
    ):
        subset = inspected_splits[split_name]
        y_pred = torch.as_tensor(diagnostics['pred_y'].to_numpy(copy=True), dtype=torch.long)
        official, _ = dataset.eval(y_pred, subset.y_array, subset.metadata_array)
        actual = diagnostics['true_y'].to_numpy()
        guessed = diagnostics['pred_y'].to_numpy()
        validation[split_name] = {
            'images': len(diagnostics),
            'official_metrics': {key: float(value) for key, value in official.items()},
            'threshold_tuned_on_this_split': split_name == 'val',
            'true_empty_fraction': float(np.mean(actual == empty_id)),
            'predicted_empty_fraction': float(np.mean(guessed == empty_id)),
            'correct_non_empty': int(np.sum(
                (actual != empty_id) & (actual == guessed)
            )),
            'non_empty_images': int(np.sum(actual != empty_id)),
            'no_box_fraction': float(np.mean(diagnostics['box_count'] == 0)),
            'box_cap_fraction': float(np.mean(
                diagnostics['box_count'] == DETECTION_MAX_BOXES
            )),
        }
    manifest = {
        'kind': 'frozen_yolo_world_exact_label_reference',
        'code': {
            'notebook': 'yolo-clip/yolo-clip-untrained.ipynb',
            'protocol_version': 'frozen-reference-v1',
            'code_cells': '1-24',
            'source_commit': None,  # Fill after committing the synchronized notebook.
        },
        'model_training_on_wilds': False,
        'checkpoint': {
            'architecture': 'Ultralytics YOLOWorld S-v2',
            'filename': WORLD_WEIGHTS, 'source': WORLD_SOURCE,
            'sha256': WORLD_SHA256, 'bytes': world_checkpoint.stat().st_size,
        },
        'dataset': {
            'name': 'iWildCam-WILDS', 'version': '2.0',
            'mirror_repository': MIRROR_REPO,
            'mirror_revision': MIRROR_REVISION,
            'archive_sha256': ARCHIVE_SHA256,
            'official_splits_used': ['train', 'id_val', 'val'],
            'final_test_splits_not_run': ['id_test', 'test'],
            'n_classes': dataset.n_classes,
            'empty_y': empty_id,
        },
        'prompts': {
            'template': PROMPT_TEMPLATE,
            'ordered_detector_index_to_wilds_y': list(animal_class_ids),
            'ordered_text': list(animal_prompts),
            'empty_excluded_from_detector': True,
        },
        'inference': {
            'image_size': WORLD_IMAGE_SIZE, 'confidence_floor': DETECTION_FLOOR,
            'iou': DETECTION_IOU, 'max_boxes': DETECTION_MAX_BOXES,
            'score': 'Ultralytics Results.boxes.conf after NMS',
            'image_rule': 'highest score; ties lower WILDS y then earlier box',
            'empty_rule': 'no boxes or top score strictly below threshold',
        },
        'threshold_selection': {
            'split': 'val', 'objective': 'WILDS-style macro-F1',
            'candidate_count': len(threshold_sweep),
            'tie_rule': 'lower threshold',
            'selected_threshold': CALIBRATED_EMPTY_THRESHOLD,
            'selected_val_macro_f1': float(selected_row['val_macro_f1']),
        },
        'validation': validation,
        'environment': versions,
        'trained_model_handoff': {
            'input': 'one iWildCam image plus official metadata',
            'output': 'exactly one WILDS class ID in 0..181 per image',
            'evaluation': 'dataset.eval on official splits; report macro-F1 and accuracy',
            'comparison': 'same splits, image size, label map, and final-test policy',
        },
        'limitations': [
            'Exact WILDS names include scientific names, broad labels, and motorcycle.',
            'Some WILDS labels are visually or textually near-duplicates.',
            'Validation threshold is selected on val; val score is not independent.',
            'Many images hit the 20-box cap; box counts are truncated.',
            'No box annotations are available here for detection mAP.',
        ],
    }
    payload = json.dumps(manifest, indent=2, sort_keys=True, ensure_ascii=False) + '\n'
    digest = hashlib.sha256(payload.encode('utf-8')).hexdigest()
    path = DATA_ROOT / f'yolo_world_reference_{digest[:12]}.json'
    if path.is_file():
        assert path.read_text(encoding='utf-8') == payload, 'Manifest path collision'
    else:
        path.write_text(payload, encoding='utf-8')
    print('Frozen reference manifest:', path)
    print('Manifest SHA-256:', digest)
    print('Threshold:', CALIBRATED_EMPTY_THRESHOLD, '| prompts:', len(animal_prompts))
    print('Recorded validation splits:', list(validation))
    print('Final test splits remain untouched.')
    return path, manifest

reference_manifest_path, reference_manifest = build_frozen_reference_manifest()

### Final-test gate (disabled)
Cell 24 documents and enforces the current boundary: no `id_test` or `test` loading in this notebook. The trained baseline notebook is still separate and unfinished. After both approaches and their comparison protocol are frozen, we can add one deliberate final-test runner and execute it once; turning this switch on now intentionally raises an error before any test access.

In [ ]:
# CELL 24
RUN_FINAL_TESTS = False
if RUN_FINAL_TESTS:
    raise RuntimeError(
        'Final test is deliberately blocked until the trained model and comparison '
        'protocol are frozen. Add a reviewed final-test runner then.'
    )
assert reference_manifest['dataset']['final_test_splits_not_run'] == [
    'id_test', 'test'
]
print('Final-test gate closed: id_test and test were not loaded or evaluated.')
print('Frozen reference handoff:', reference_manifest_path)